# Fase 1 — Análisis Exploratorio de Datos (EDA)
## Dataset: Kuzushiji-MNIST (KMNIST)

**Objetivo:** Comprender la estructura del dataset antes de cualquier preprocesamiento.

| Aspecto | Detalle |
|---|---|
| Imágenes totales | 70,000 (60k train / 10k test) |
| Clases | 10 caracteres japoneses (kuzushiji) |
| Resolución | 28×28 píxeles, escala de grises |
| Rango de valores | uint8 → [0, 255] |

## 0. Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
import sys

# Agregar raíz del proyecto al path para importar funciones.py
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from funciones import cargar_kmnist, plot_muestras_por_clase, plot_distribucion_clases, plot_histogramas_intensidad

DATA_DIR  = ROOT / 'data'
OUT_DIR   = ROOT / 'outputs' / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Nombres de las 10 clases (romanización)
CLASES = ['お (o)', 'き (ki)', 'す (su)', 'つ (tsu)', 'な (na)',
          'は (ha)', 'ま (ma)', 'や (ya)', 'れ (re)', 'を (wo)']

sns.set_theme(style='whitegrid', palette='tab10')
print('✅ Imports OK')

## 1. Carga del dataset

In [ ]:
X_train, y_train, X_test, y_test = cargar_kmnist(DATA_DIR)

print(f'Train → imágenes: {X_train.shape},  labels: {y_train.shape}')
print(f'Test  → imágenes: {X_test.shape},   labels: {y_test.shape}')
print(f'\nTipo de dato : {X_train.dtype}')
print(f'Rango valores: [{X_train.min()}, {X_train.max()}]')
print(f'Clases únicas: {np.unique(y_train)}')

## 2. Distribución de clases

**¿Por qué importa?** Un dataset desbalanceado fuerza a usar métricas como F1-macro en lugar de accuracy. Si una clase tiene pocas muestras, los modelos tenderán a ignorarla.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_distribucion_clases(y_train, y_test, CLASES, axes)
plt.tight_layout()
plt.savefig(OUT_DIR / '01_distribucion_clases.png', dpi=150)
plt.show()
print('\nConteo por clase (train):')
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f'  Clase {cls} ({CLASES[cls]}): {cnt} imágenes')

## 3. Visualización de muestras por clase

**¿Por qué importa?** Identificar variabilidad intra-clase (mismo carácter con distintos estilos de escritura) y similitud inter-clase (caracteres visualmente parecidos = problema difícil).

In [ ]:
fig = plot_muestras_por_clase(X_train, y_train, CLASES, n_muestras=8)
plt.savefig(OUT_DIR / '02_muestras_por_clase.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Imagen promedio por clase

**¿Por qué importa?** La imagen promedio revela la estructura "típica" de cada carácter. Si dos promedios son muy similares, esas clases serán difíciles de separar con features simples.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()

for cls in range(10):
    mask = y_train == cls
    img_promedio = X_train[mask].mean(axis=0)
    axes[cls].imshow(img_promedio, cmap='gray')
    axes[cls].set_title(f'Clase {cls}\n{CLASES[cls]}', fontsize=9)
    axes[cls].axis('off')

fig.suptitle('Imagen Promedio por Clase', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / '03_imagen_promedio.png', dpi=150)
plt.show()

## 5. Histogramas de intensidad de píxeles

**¿Por qué importa?** KMNIST tiene fondo negro (0) y trazos blancos (255). El histograma confirmará esto y justificará el uso de umbralización de Otsu en la Fase 2.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_histogramas_intensidad(X_train, y_train, CLASES, axes)
plt.tight_layout()
plt.savefig(OUT_DIR / '04_histogramas_intensidad.png', dpi=150)
plt.show()

## 6. Varianza por píxel (mapa de calor)

**¿Por qué importa?** Las zonas de alta varianza son las más informativas para la clasificación. Las zonas de baja varianza (siempre negras = bordes) podrían descartarse.

In [ ]:
varianza_pixeles = X_train.var(axis=0)  # shape (28, 28)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(varianza_pixeles, cmap='hot')
fig.colorbar(im, ax=ax, label='Varianza')
ax.set_title('Varianza por Píxel (todos los ejemplos)', fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig(OUT_DIR / '05_varianza_pixeles.png', dpi=150)
plt.show()

pct_baja_var = (varianza_pixeles < 100).mean() * 100
print(f'Píxeles con varianza < 100: {pct_baja_var:.1f}% del total')

## 7. Análisis estadístico de intensidad por clase

In [ ]:
stats = []
for cls in range(10):
    imgs = X_train[y_train == cls].astype(float)
    stats.append({
        'Clase': f'{cls} ({CLASES[cls]})',
        'Media':   imgs.mean().round(2),
        'Std':     imgs.std().round(2),
        'Min':     imgs.min(),
        'Max':     imgs.max(),
        'Píxeles > 127 (%)': (imgs > 127).mean().round(4) * 100
    })

df_stats = pd.DataFrame(stats)
df_stats

## 8. Conclusiones del EDA

Completa con tus observaciones después de correr el notebook:

| Aspecto | Observación |
|---|---|
| Balanceo | ¿El dataset es balanceado? |
| Distribución de píxeles | ¿Bimodal (fondo/trazo)? |
| Clases visualmente similares | ¿Cuáles son difíciles de distinguir? |
| Variabilidad intra-clase | ¿Alta o baja variación de escritura? |
| Zonas informativas | ¿Qué zona del mapa de calor es más informativa? |